# SS3 Part D evidence notebook (STADIOEquities alert triage)
Does NOT touch the submitted SS3 Part B notebook. It rebuilds the client data
with leakage safe preprocessing (imputation/scaling fitted inside each fold) and answers the evidence
questions behind Part D. 

Note: numbers will not exactly match SS3 Part B, because this pipeline differs (see cell 3).

In [3]:
# 1. Setup
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

DATA_DIR = "../data/raw_ss3"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

In [ ]:
# 2. Load and clean 
def load(name):
    return pd.read_csv(f"{DATA_DIR}/{name}")

def cat(s):
    """Trim, lower-case, and make missing values real NaN (so imputers work)."""
    s = s.astype("string").str.strip().str.lower().str.replace("_", " ")
    return s.astype(object).where(s.notna(), np.nan)

def ts(s):
    return pd.to_datetime(s, format="mixed", errors="coerce")

alerts = load("table1_compliance_alerts.csv").drop_duplicates("alert_id").copy()
acct   = load("table2_account_funding_profile.csv").drop_duplicates("account_id").copy()
demo   = load("table3_client_demographics.csv").drop_duplicates("account_id").copy()
tx     = load("table4_transaction_history.csv").drop_duplicates("transaction_id").copy()
sess   = load("table5_app_web_sessions.csv").copy()

alerts["alert_timestamp"] = ts(alerts["alert_timestamp"])
alerts["investigation_disposition"] = cat(alerts["investigation_disposition"])
alerts["rule_triggered"] = cat(alerts["rule_triggered"])
alerts["assigned_investigator_id_anonymised"] = cat(alerts["assigned_investigator_id_anonymised"])
acct["registration_date"] = ts(acct["registration_date"])
for c in ["kyc_status", "account_status", "premium_subscription_status"]:
    acct[c] = cat(acct[c])
demo["age"] = pd.to_numeric(demo["age"], errors="coerce")
demo.loc[~demo["age"].between(18, 100), "age"] = np.nan          # out-of-range ages -> missing
for c in ["province_region", "stated_investment_goal", "stated_risk_appetite"]:
    demo[c] = cat(demo[c])
tx["transaction_timestamp"] = ts(tx["transaction_timestamp"])
tx["transaction_type"] = cat(tx["transaction_type"]); tx["channel"] = cat(tx["channel"])
tx["amount"] = pd.to_numeric(tx["amount"], errors="coerce").clip(lower=0)
sess["session_start_timestamp"] = ts(sess["session_start_timestamp"])
print("Alerts after de-duplication:", len(alerts), "| unparseable alert timestamps:", alerts["alert_timestamp"].isna().sum())

Alerts after de-duplication: 2864 | unparseable alert timestamps: 0


In [5]:
# 3. Evidence 1: date range, drift, account repetition, join quality
print("Earliest alert:", alerts["alert_timestamp"].min(), "| Latest alert:", alerts["alert_timestamp"].max())
known = alerts[alerts["investigation_disposition"].isin(["true positive", "false positive"])].copy()
known["is_tp"] = (known["investigation_disposition"] == "true positive").astype(int)
print("\nLabelled alerts:", len(known), "| true positives:", known["is_tp"].sum(), "| share:", round(known["is_tp"].mean(), 4))
print("\nTrue-positive rate by quarter (drift check):")
print(known.groupby(known["alert_timestamp"].dt.to_period("Q"))["is_tp"].agg(["count", "mean"]).round(3))
print("\nDistinct accounts among alerts:", alerts["account_id"].nunique())
print("Alerts per account:"); print(alerts.groupby("account_id").size().describe().round(2))
print("\nBlank transaction_id share (README says ~28%):", round(alerts["transaction_id"].isna().mean(), 3))
print("Non-blank transaction_ids that exist in table 4:", round(alerts["transaction_id"].dropna().isin(tx["transaction_id"]).mean(), 3))
print("\nAlert share by channel / transaction_type (new or rare channels show here):")
j = alerts.merge(tx[["transaction_id", "channel", "transaction_type"]], on="transaction_id", how="left")
print(j["channel"].value_counts(dropna=False)); print(j["transaction_type"].value_counts(dropna=False))

Earliest alert: 2022-10-30 02:04:23 | Latest alert: 2026-09-04 14:39:28

Labelled alerts: 2587 | true positives: 351 | share: 0.1357

True-positive rate by quarter (drift check):
                 count   mean
alert_timestamp              
2022Q4               9  0.000
2023Q1              47  0.128
2023Q2              88  0.239
2023Q3             115  0.139
2023Q4             117  0.120
2024Q1             128  0.180
2024Q2             137  0.102
2024Q3             147  0.170
2024Q4             165  0.133
2025Q1             205  0.112
2025Q2             163  0.104
2025Q3             198  0.106
2025Q4             234  0.154
2026Q1             234  0.137
2026Q2             295  0.129
2026Q3             305  0.141

Distinct accounts among alerts: 1434
Alerts per account:
count    1434.00
mean        2.00
std         1.12
min         1.00
25%         1.00
50%         2.00
75%         3.00
max         5.00
dtype: float64

Blank transaction_id share (README says ~28%): 0.416
Non-blank transact

In [6]:
# 4. Build the modelling table (features use only information available BEFORE each alert)
d = known.merge(acct[["account_id", "registration_date", "kyc_status", "account_status", "premium_subscription_status"]], on="account_id", how="left")
d = d.merge(demo[["account_id", "age", "province_region", "stated_investment_goal", "stated_risk_appetite"]], on="account_id", how="left")
d = d.merge(tx[["transaction_id", "transaction_type", "channel", "amount", "portfolio_concentration_at_transaction"]], on="transaction_id", how="left")
d["amount_log"] = np.log1p(d["amount"])
d["account_age_days"] = (d["alert_timestamp"] - d["registration_date"]).dt.days

# prior transaction behaviour (strictly before the alert)
m = d[["alert_id", "account_id", "alert_timestamp"]].merge(
    tx[["account_id", "transaction_timestamp", "transaction_type", "amount"]], on="account_id", how="left")
m = m[m["transaction_timestamp"] < m["alert_timestamp"]]
beh = pd.DataFrame({"prior_tx_count": m.groupby("alert_id").size(),
                    "prior_tx_log_max_amount": np.log1p(m.groupby("alert_id")["amount"].max()),
                    "prior_withdrawal_count": m[m["transaction_type"].astype(str).str.contains("withdraw")].groupby("alert_id").size()})
d = d.merge(beh, left_on="alert_id", right_index=True, how="left")
for c in ["prior_tx_count", "prior_withdrawal_count"]:
    d[c] = d[c].fillna(0)
# prior alerts on the same account (all alerts, earlier timestamp)
srt = alerts.sort_values("alert_timestamp")
srt["prior_alert_count"] = srt.groupby("account_id").cumcount()
d = d.merge(srt[["alert_id", "prior_alert_count"]], on="alert_id", how="left")

# prior session behaviour (strictly before the alert)
s2 = d[["alert_id", "account_id", "alert_timestamp"]].merge(
    sess[["account_id", "session_start_timestamp", "device_identifier", "login_region_country"]], on="account_id", how="left")
s2 = s2[s2["session_start_timestamp"] < s2["alert_timestamp"]]
sb = pd.DataFrame({"prior_session_count": s2.groupby("alert_id").size(),
                   "prior_distinct_devices": s2.groupby("alert_id")["device_identifier"].nunique(),
                   "prior_distinct_countries": s2.groupby("alert_id")["login_region_country"].nunique()})
d = d.merge(sb, left_on="alert_id", right_index=True, how="left")
for c in sb.columns:
    d[c] = d[c].fillna(0)

d = d.sort_values("alert_timestamp").reset_index(drop=True)
y = d["is_tp"].values.astype(int)
groups = d["account_id"].values

BASE_NUM = ["rule_assigned_severity_score", "age", "amount_log", "portfolio_concentration_at_transaction"]
BASE_CAT = ["rule_triggered", "kyc_status", "account_status", "premium_subscription_status", "province_region",
            "stated_risk_appetite", "stated_investment_goal", "transaction_type", "channel"]
BEH_NUM = ["prior_tx_count", "prior_tx_log_max_amount", "prior_withdrawal_count", "prior_alert_count", "account_age_days"]
SES_NUM = ["prior_session_count", "prior_distinct_devices", "prior_distinct_countries"]
FEATURE_SETS = {"base": (BASE_NUM, BASE_CAT),
                "base+behaviour": (BASE_NUM + BEH_NUM, BASE_CAT),
                "base+behaviour+sessions": (BASE_NUM + BEH_NUM + SES_NUM, BASE_CAT)}
ALL_NUM, ALL_CAT = FEATURE_SETS["base+behaviour+sessions"]
print("Modelling table:", d.shape, "| positives:", y.sum(), "| base rate:", round(y.mean(), 4))

Modelling table: (2587, 32) | positives: 351 | base rate: 0.1357


In [7]:
# 5. Pipelines and evaluation helpers
def make_pre(num, catc):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), catc)])

def lr_model(num, catc):
    return Pipeline([("pre", make_pre(num, catc)), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])

def rf_weighted(num, catc):   # no SMOTE: class weights + min leaf size to limit overfitting
    return Pipeline([("pre", make_pre(num, catc)), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=0))])

def rf_smote_d20(num, catc):  # closest to the SS3 Part B setup (SMOTE inside each training fold, depth 20)
    return ImbPipeline([("pre", make_pre(num, catc)), ("smote", SMOTE(random_state=0)), ("clf", RandomForestClassifier(
        n_estimators=100, max_depth=20, n_jobs=-1, random_state=0))])

def metrics(yt, p):
    k = max(1, int(round(0.10 * len(p)))); top = np.argsort(-p)[:k]
    return {"roc_auc": float(roc_auc_score(yt, p)), "pr_auc": float(average_precision_score(yt, p)),
            "precision_top10pct": float(yt[top].mean()), "lift_top10pct": float(yt[top].mean() / max(yt.mean(), 1e-9))}

def cv_scores(factory, fs, seeds=(0, 1, 2), frac=1.0):
    num, catc = fs; cols = num + catc; rows = []
    for seed in seeds:
        for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(d, y, groups):
            if frac < 1:
                tr, _ = train_test_split(tr, train_size=frac, stratify=y[tr], random_state=seed)
            mdl = factory(num, catc).fit(d.iloc[tr][cols], y[tr])
            rows.append(metrics(y[te], mdl.predict_proba(d.iloc[te][cols])[:, 1]))
    return pd.DataFrame(rows)

def summarise(df):
    return df.agg(["mean", "std"]).round(3)
print("Random-chance reference: ROC-AUC 0.5, PR-AUC ~", round(y.mean(), 3), "(= base rate), lift 1.0")

Random-chance reference: ROC-AUC 0.5, PR-AUC ~ 0.136 (= base rate), lift 1.0


In [8]:
# 6. Evidence 2: models x feature sets (grouped, repeated CV; 5 folds x 3 seeds)
results = {}
for fs_name, fs in FEATURE_SETS.items():
    for mname, fac in [("LogReg", lr_model), ("RF weighted", rf_weighted), ("RF SMOTE depth20", rf_smote_d20)]:
        results[(fs_name, mname)] = summarise(cv_scores(fac, fs)).loc["mean"]
        print("done:", fs_name, mname)
comp = pd.DataFrame(results).T.round(3); comp.index.names = ["features", "model"]
print("\nMean over 15 grouped CV folds (higher is better):"); print(comp)
sd = cv_scores(rf_weighted, FEATURE_SETS["base+behaviour+sessions"]).std().round(3)
print("\nFold-to-fold std for RF weighted, all features:"); print(sd)

done: base LogReg
done: base RF weighted
done: base RF SMOTE depth20
done: base+behaviour LogReg
done: base+behaviour RF weighted
done: base+behaviour RF SMOTE depth20
done: base+behaviour+sessions LogReg
done: base+behaviour+sessions RF weighted
done: base+behaviour+sessions RF SMOTE depth20

Mean over 15 grouped CV folds (higher is better):
                                          roc_auc  pr_auc  precision_top10pct  lift_top10pct
features                model                                                               
base                    LogReg              0.575   0.174               0.192          1.418
                        RF weighted         0.570   0.177               0.201          1.484
                        RF SMOTE depth20    0.529   0.153               0.150          1.106
base+behaviour          LogReg              0.615   0.210               0.255          1.881
                        RF weighted         0.637   0.222               0.251          1.852
    

In [9]:
# 7. Evidence 3: learning curve (does more data help?)
lc = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    r = cv_scores(rf_weighted, FEATURE_SETS["base+behaviour+sessions"], frac=frac).mean()
    lc.append({"train_fraction": frac, "approx_train_alerts": int(frac * len(d) * 0.8), **r.round(3).to_dict()})
print(pd.DataFrame(lc))
print("If the curve is still rising at 1.0, more data would likely help; if flat, more of the same data will not.")

   train_fraction  approx_train_alerts  roc_auc  pr_auc  precision_top10pct  lift_top10pct
0            0.25                  517    0.616   0.203               0.214          1.578
1            0.50                 1034    0.628   0.213               0.246          1.814
2            0.75                 1552    0.634   0.224               0.241          1.777
3            1.00                 2069    0.642   0.224               0.247          1.824
If the curve is still rising at 1.0, more data would likely help; if flat, more of the same data will not.


In [10]:
# 8. Evidence 4: drift. Random split vs time split vs rolling quarters
cut = int(len(d) * 0.7)
tr, te = np.arange(cut), np.arange(cut, len(d))
cols = ALL_NUM + ALL_CAT
mt = rf_weighted(ALL_NUM, ALL_CAT).fit(d.iloc[tr][cols], y[tr])
tmetric = metrics(y[te], mt.predict_proba(d.iloc[te][cols])[:, 1])
rand = []
for seed in range(5):
    a, b = train_test_split(np.arange(len(d)), test_size=0.3, stratify=y, random_state=seed)
    mr = rf_weighted(ALL_NUM, ALL_CAT).fit(d.iloc[a][cols], y[a])
    rand.append(metrics(y[b], mr.predict_proba(d.iloc[b][cols])[:, 1]))
print("TIME split (train oldest 70%, test newest 30%):", {k: round(v, 3) for k, v in tmetric.items()})
print("RANDOM split (mean of 5):", pd.DataFrame(rand).mean().round(3).to_dict())
print("TP rate train vs test:", round(y[tr].mean(), 3), "vs", round(y[te].mean(), 3))
print("\nRolling forward: train on everything before each quarter, test on that quarter")
q = d["alert_timestamp"].dt.to_period("Q"); roll = []
for per in sorted(q.unique()):
    tr_i = np.where(q < per)[0]; te_i = np.where(q == per)[0]
    if len(tr_i) < 500 or len(te_i) < 80 or y[te_i].sum() < 8 or y[tr_i].sum() < 30:
        continue
    mq = rf_weighted(ALL_NUM, ALL_CAT).fit(d.iloc[tr_i][cols], y[tr_i])
    roll.append({"quarter": str(per), "n_test": len(te_i), "tp_rate": round(y[te_i].mean(), 3),
                 **{k: round(v, 3) for k, v in metrics(y[te_i], mq.predict_proba(d.iloc[te_i][cols])[:, 1]).items()}})
print(pd.DataFrame(roll))

TIME split (train oldest 70%, test newest 30%): {'roc_auc': 0.616, 'pr_auc': 0.191, 'precision_top10pct': 0.192, 'lift_top10pct': 1.437}
RANDOM split (mean of 5): {'roc_auc': 0.642, 'pr_auc': 0.223, 'precision_top10pct': 0.238, 'lift_top10pct': 1.765}
TP rate train vs test: 0.136 vs 0.134

Rolling forward: train on everything before each quarter, test on that quarter
  quarter  n_test  tp_rate  roc_auc  pr_auc  precision_top10pct  lift_top10pct
0  2024Q2     137    0.102    0.689   0.266               0.214          2.097
1  2024Q3     147    0.170    0.696   0.325               0.333          1.960
2  2024Q4     165    0.133    0.653   0.228               0.188          1.406
3  2025Q1     205    0.112    0.658   0.189               0.250          2.228
4  2025Q2     163    0.104    0.463   0.099               0.000          0.000
5  2025Q3     198    0.106    0.617   0.150               0.150          1.414
6  2025Q4     234    0.154    0.744   0.382               0.522          3.39

In [11]:
# 9. Evidence 5: unseen category (proxy for a new payment method or channel)
oof = np.zeros(len(d))
for tr_i, te_i in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0).split(d, y, groups):
    oof[te_i] = rf_weighted(ALL_NUM, ALL_CAT).fit(d.iloc[tr_i][cols], y[tr_i]).predict_proba(d.iloc[te_i][cols])[:, 1]
rows = []
for col in ["channel", "transaction_type"]:
    for v in d[col].dropna().unique():
        te_i = np.where(d[col] == v)[0]; tr_i = np.where(d[col] != v)[0]
        if len(te_i) < 40 or y[te_i].sum() < 8 or (len(te_i) - y[te_i].sum()) < 8:
            continue
        mh = rf_weighted(ALL_NUM, ALL_CAT).fit(d.iloc[tr_i][cols], y[tr_i])
        rows.append({"column": col, "category_held_out": v, "n": len(te_i),
                     "roc_auc_normal_cv": round(roc_auc_score(y[te_i], oof[te_i]), 3),
                     "roc_auc_when_unseen": round(roc_auc_score(y[te_i], mh.predict_proba(d.iloc[te_i][cols])[:, 1]), 3)})
print(pd.DataFrame(rows) if rows else "No category had enough alerts/positives for this test.")
print("Small samples: treat differences of a few hundredths as noise.")

             column category_held_out    n  roc_auc_normal_cv  roc_auc_when_unseen
0           channel               web  405              0.592                0.587
1           channel               app  683              0.630                0.647
2           channel               eft  264              0.784                0.785
3           channel              card  174              0.668                0.699
4  transaction_type              sell  260              0.692                0.739
5  transaction_type               buy  442              0.637                0.631
6  transaction_type           deposit  513              0.622                0.652
7  transaction_type        withdrawal  202              0.687                0.727
8  transaction_type               fee  109              0.707                0.749
Small samples: treat differences of a few hundredths as noise.


In [12]:
# 10. Evidence 6: does class_weight='balanced' on SMOTE-balanced data change anything?
a, b = train_test_split(np.arange(len(d)), test_size=0.2, stratify=y, random_state=42)
pre = make_pre(ALL_NUM, ALL_CAT).fit(d.iloc[a][cols]); Xa = pre.transform(d.iloc[a][cols]); Xb = pre.transform(d.iloc[b][cols])
Xs, ys = SMOTE(random_state=42).fit_resample(Xa, y[a])
print("Balanced training counts:", np.bincount(ys))
base_rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1).fit(Xs, ys)
wt_rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1, class_weight="balanced").fit(Xs, ys)
pb, pw = base_rf.predict_proba(Xb)[:, 1], wt_rf.predict_proba(Xb)[:, 1]
print("Identical probabilities?", np.allclose(pb, pw), "| max probability difference:", round(float(np.abs(pb - pw).max()), 4))
print("Test predictions that differ at the default 0.5 threshold:", int((base_rf.predict(Xb) != wt_rf.predict(Xb)).sum()))
print("Weights are 1.0 on a 50/50 set, so class_weight adds no information. If the forests still differ, treat that as build-to-build randomness, not a real effect.")
print("Train-set performance (overfitting check):", {k: round(v, 3) for k, v in metrics(ys, base_rf.predict_proba(Xs)[:, 1]).items()})
print("Test-set performance:", {k: round(v, 3) for k, v in metrics(y[b], pb).items()})

Balanced training counts: [1788 1788]
Identical probabilities? False | max probability difference: 0.215
Test predictions that differ at the default 0.5 threshold: 3
Weights are 1.0 on a 50/50 set, so class_weight adds no information. If the forests still differ, treat that as build-to-build randomness, not a real effect.
Train-set performance (overfitting check): {'roc_auc': 1.0, 'pr_auc': 1.0, 'precision_top10pct': 1.0, 'lift_top10pct': 2.0}
Test-set performance: {'roc_auc': 0.688, 'pr_auc': 0.255, 'precision_top10pct': 0.346, 'lift_top10pct': 2.562}


In [13]:
# 11. Evidence 7: investigator label noise (28 investigators)
inv = d.groupby("assigned_investigator_id_anonymised")["is_tp"].agg(n="count", tp_rate="mean", tps="sum")
rule_rate = d.groupby("rule_triggered")["is_tp"].mean()
d["_exp"] = d["rule_triggered"].map(rule_rate)
inv["expected_tps_given_rule_mix"] = d.groupby("assigned_investigator_id_anonymised")["_exp"].sum()
inv["actual_vs_expected"] = inv["tps"] / inv["expected_tps_given_rule_mix"]
print(inv.sort_values("tp_rate").round(3))
chi2, p, dof, _ = chi2_contingency(pd.crosstab(d["assigned_investigator_id_anonymised"], d["is_tp"]))
print(f"\nChi-square test, investigator vs outcome: chi2={chi2:.1f}, dof={dof}, p={p:.4f}")
print("Range of TP rate:", round(inv['tp_rate'].min(), 3), "to", round(inv['tp_rate'].max(), 3))
print("actual_vs_expected far from 1.0 means the difference is not explained by which rules they were given.")

                                       n  tp_rate  tps  expected_tps_given_rule_mix  actual_vs_expected
assigned_investigator_id_anonymised                                                                    
inv-018                               95    0.021    2                       12.976               0.154
inv-004                               97    0.031    3                       12.779               0.235
inv-003                               79    0.038    3                       10.776               0.278
inv-006                               76    0.039    3                       10.562               0.284
inv-011                              103    0.058    6                       13.883               0.432
inv-023                               87    0.080    7                       11.744               0.596
inv-002                               87    0.092    8                       12.005               0.666
inv-028                               95    0.095    9          